# Skin Disease 0.99 Sprint Final (New Notebook)

This is a fully new notebook with a stable high-score strategy:

1. Multi-seed x multi-model CV ensemble  
2. OOF weight optimization  
3. Stacking (Logistic Regression on OOF probabilities)  
4. Bias + temperature calibration (OOF-optimized)  
5. Consensus pseudo-label refinement (conservative)  
6. Retrieval-based near-duplicate correction (feature cosine similarity)  


In [ ]:
import copy
import gc
import json
import random
import uuid
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from torchvision import models
from torchvision.transforms import v2

print('torch:', torch.__version__)


In [ ]:
# =========================
# Paths / config
# =========================
LOCAL_BASE = Path('/Users/songling/Desktop/Skin Disease Classification')
PLATFORM_BASE = Path('dataset/public')

if PLATFORM_BASE.exists():
    MODE = 'platform'
    BASE_DIR = PLATFORM_BASE
    OUTPUT_DIR = Path('working')
elif LOCAL_BASE.exists():
    MODE = 'local'
    BASE_DIR = LOCAL_BASE
    OUTPUT_DIR = BASE_DIR
else:
    raise FileNotFoundError('Cannot find dataset directory.')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR = OUTPUT_DIR / 'runs'
RUNS_DIR.mkdir(parents=True, exist_ok=True)
run_name = datetime.now().strftime('run_99final_%Y%m%d_%H%M%S')
run_dir = RUNS_DIR / run_name
run_dir.mkdir(parents=True, exist_ok=False)

TRAIN_CSV = BASE_DIR / 'train.csv'
TEST_CSV = BASE_DIR / 'test.csv'
TRAIN_IMG_DIR = BASE_DIR / 'train'
TEST_IMG_DIR = BASE_DIR / 'test'

CLASS_NAMES = ['acne', 'eksim', 'herpes', 'panu', 'rosacea']
label2idx = {c: i for i, c in enumerate(CLASS_NAMES)}
idx2label = {i: c for c, i in label2idx.items()}
N_CLASSES = len(CLASS_NAMES)

SEEDS = [42, 2024]
N_SPLITS = 5
IMG_SIZE = 224
BATCH_SIZE = 16

EPOCHS = 14
HEAD_EPOCHS = 2
LR_HEAD = 1e-3
LR_FINE = 2e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
EARLY_STOP = 4
GRAD_CLIP_NORM = 1.0

MODEL_CONFIGS = [
    {'name': 'swin_v2_t', 'base_weight': 0.45},
    {'name': 'efficientnet_v2_s', 'base_weight': 0.35},
    {'name': 'convnext_small', 'base_weight': 0.20},
]

# Conservative pseudo-label settings
ENABLE_PSEUDO = True
PSEUDO_THRESHOLD = 0.995
PSEUDO_MARGIN = 0.25
PSEUDO_MIN_COUNT = 12
PSEUDO_BLEND_ALPHA = 0.15

# Retrieval correction settings
ENABLE_RETRIEVAL_CORRECTION = True
RETRIEVAL_SIM_THRESHOLD = 0.93
RETRIEVAL_BLEND = 0.25

NUM_WORKERS = 0 if MODE == 'local' else 2
PERSISTENT_WORKERS = NUM_WORKERS > 0

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

PIN_MEMORY = (device.type == 'cuda')
USE_AMP = (device.type == 'cuda')

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

_available_model_map = {
    'swin_v2_t': hasattr(models, 'swin_v2_t'),
    'efficientnet_v2_s': hasattr(models, 'efficientnet_v2_s'),
    'convnext_small': hasattr(models, 'convnext_small'),
}
MODEL_CONFIGS = [m for m in MODEL_CONFIGS if _available_model_map.get(m['name'], False)]
if len(MODEL_CONFIGS) == 0:
    raise RuntimeError('No supported models available in this environment.')

print('Mode:', MODE)
print('Device:', device)
print('Models:', [m['name'] for m in MODEL_CONFIGS])
print('Run dir:', run_dir)


In [ ]:
# =========================
# Data / transforms / datasets
# =========================
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

assert set(train_df['disease'].unique()) == set(CLASS_NAMES), 'Unexpected class names'
assert len(test_df) == 180, f'Test row count mismatch: {len(test_df)}'

train_df['label'] = train_df['disease'].map(label2idx)


def make_class_weights(df):
    counts = df['label'].value_counts().sort_index().values
    return torch.tensor(len(df) / (N_CLASSES * counts), dtype=torch.float32)


class AddGaussianNoise(nn.Module):
    def __init__(self, std=0.02, p=0.25):
        super().__init__()
        self.std = std
        self.p = p

    def forward(self, x):
        if torch.rand(1).item() < self.p:
            x = torch.clamp(x + torch.randn_like(x) * self.std, 0.0, 1.0)
        return x


MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_tfms = v2.Compose([
    v2.Resize((IMG_SIZE, IMG_SIZE)),
    v2.RandomHorizontalFlip(0.5),
    v2.RandomVerticalFlip(0.15),
    v2.RandomRotation(20),
    v2.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.9, 1.1)),
    v2.ColorJitter(brightness=0.18, contrast=0.18, saturation=0.12, hue=0.03),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    AddGaussianNoise(std=0.02, p=0.25),
    v2.Normalize(MEAN, STD),
])

valid_tfms = v2.Compose([
    v2.Resize((IMG_SIZE, IMG_SIZE)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])


class SkinDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(self.image_dir / row['filename']).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        if self.is_test:
            return image, int(row['id'])
        return image, int(row['label'])


def build_loader(ds, shuffle):
    kwargs = dict(batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    if PERSISTENT_WORKERS:
        kwargs['persistent_workers'] = True
    return DataLoader(ds, **kwargs)


In [ ]:
# =========================
# Model / inference utilities
# =========================
def build_model(model_name):
    if model_name == 'swin_v2_t':
        try:
            m = models.swin_v2_t(weights=models.Swin_V2_T_Weights.IMAGENET1K_V1)
        except Exception as e:
            print('Warning: swin_v2_t pretrained unavailable:', e)
            m = models.swin_v2_t(weights=None)
        in_features = m.head.in_features
        m.head = nn.Linear(in_features, N_CLASSES)
    elif model_name == 'efficientnet_v2_s':
        try:
            m = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
        except Exception as e:
            print('Warning: efficientnet_v2_s pretrained unavailable:', e)
            m = models.efficientnet_v2_s(weights=None)
        in_features = m.classifier[1].in_features
        m.classifier[1] = nn.Linear(in_features, N_CLASSES)
    elif model_name == 'convnext_small':
        try:
            m = models.convnext_small(weights=models.ConvNeXt_Small_Weights.IMAGENET1K_V1)
        except Exception as e:
            print('Warning: convnext_small pretrained unavailable:', e)
            m = models.convnext_small(weights=None)
        in_features = m.classifier[2].in_features
        m.classifier[2] = nn.Linear(in_features, N_CLASSES)
    else:
        raise ValueError(model_name)

    return m.to(device)


def set_head_only(model, model_name, head_only):
    for p in model.parameters():
        p.requires_grad = not head_only

    if head_only:
        if model_name == 'swin_v2_t':
            for p in model.head.parameters():
                p.requires_grad = True
        elif model_name == 'efficientnet_v2_s':
            for p in model.classifier[1].parameters():
                p.requires_grad = True
        elif model_name == 'convnext_small':
            for p in model.classifier[2].parameters():
                p.requires_grad = True


def pad_or_crop(x, h_target, w_target):
    _, _, h, w = x.shape
    if h > h_target:
        top = (h - h_target) // 2
        x = x[:, :, top:top + h_target, :]
    elif h < h_target:
        p = h_target - h
        x = F.pad(x, (0, 0, p // 2, p - p // 2), mode='reflect')

    _, _, h, w = x.shape
    if w > w_target:
        left = (w - w_target) // 2
        x = x[:, :, :, left:left + w_target]
    elif w < w_target:
        p = w_target - w
        x = F.pad(x, (p // 2, p - p // 2, 0, 0), mode='reflect')
    return x


def scale_view(x, scale):
    _, _, h, w = x.shape
    y = F.interpolate(x, scale_factor=scale, mode='bilinear', align_corners=False)
    return pad_or_crop(y, h, w)


@torch.no_grad()
def tta_logits_8(model, x):
    views = [
        x,
        torch.flip(x, dims=[3]),
        torch.flip(x, dims=[2]),
        torch.flip(x, dims=[2, 3]),
        scale_view(x, 0.92),
        scale_view(x, 1.08),
        torch.flip(scale_view(x, 0.92), dims=[3]),
        torch.flip(scale_view(x, 1.08), dims=[2]),
    ]
    out = 0
    for v in views:
        out = out + model(v)
    return out / len(views)


@torch.no_grad()
def predict_proba(model, loader, use_tta=True):
    model.eval()
    probs = []
    for batch in loader:
        images = batch[0].to(device, non_blocking=True)
        logits = tta_logits_8(model, images) if use_tta else model(images)
        probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(probs, axis=0)


def macro_f1(y_true, proba):
    return f1_score(y_true, np.argmax(proba, axis=1), average='macro')


def optimize_weights(y_true, member_oof_list, base_w=None, n_trials=2500, seed=123):
    rng = np.random.default_rng(seed)
    n = len(member_oof_list)

    if base_w is None:
        base_w = np.ones(n, dtype=np.float32) / n
    else:
        base_w = np.array(base_w, dtype=np.float32)
        base_w = base_w / base_w.sum()

    def blend_score(w):
        p = np.zeros_like(member_oof_list[0])
        for wi, pi in zip(w, member_oof_list):
            p += wi * pi
        return macro_f1(y_true, p)

    best_w = base_w.copy()
    best_s = blend_score(best_w)

    for _ in range(n_trials):
        w = rng.dirichlet(np.ones(n)).astype(np.float32)
        s = blend_score(w)
        if s > best_s:
            best_s = s
            best_w = w

    return best_w, float(best_s)


def optimize_bias_temp(y_true, proba, rounds=4):
    eps = 1e-8
    logp = np.log(np.clip(proba, eps, 1.0))
    bias = np.zeros(N_CLASSES, dtype=np.float32)
    temp = 1.0

    def apply(b, t):
        z = (logp + b[None, :]) / t
        z = z - z.max(axis=1, keepdims=True)
        e = np.exp(z)
        return e / e.sum(axis=1, keepdims=True)

    best = macro_f1(y_true, apply(bias, temp))

    for _ in range(rounds):
        improved = False
        for c in range(N_CLASSES):
            for step in [0.25, 0.12, 0.06, 0.03]:
                for d in (-step, step):
                    b2 = bias.copy()
                    b2[c] += d
                    s = macro_f1(y_true, apply(b2, temp))
                    if s > best:
                        best = s
                        bias = b2
                        improved = True
        for step in [0.15, 0.08, 0.04]:
            for d in (-step, step):
                t2 = max(0.55, min(1.75, temp + d))
                s = macro_f1(y_true, apply(bias, t2))
                if s > best:
                    best = s
                    temp = t2
                    improved = True
        if not improved:
            break

    return bias, float(temp), float(best)


def apply_bias_temp(proba, bias, temp):
    eps = 1e-8
    z = (np.log(np.clip(proba, eps, 1.0)) + bias[None, :]) / temp
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


In [ ]:
# =========================
# Stage A: train members and collect OOF/Test probabilities
# =========================
y_true = train_df['label'].values
member_oof_list = []
member_test_list = []
member_name_list = []
member_logs = []

for seed in SEEDS:
    seed_everything(seed)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

    for mcfg in MODEL_CONFIGS:
        model_name = mcfg['name']
        print('')
        print('Seed', seed, '| Model', model_name)

        oof_member = np.zeros((len(train_df), N_CLASSES), dtype=np.float32)
        test_member = np.zeros((len(test_df), N_CLASSES), dtype=np.float32)

        for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df['label']), start=1):
            print('Fold', fold, '/', N_SPLITS)

            tr_df = train_df.iloc[tr_idx].copy()
            va_df = train_df.iloc[va_idx].copy()
            y_val = va_df['label'].values

            tr_ds = SkinDataset(tr_df, TRAIN_IMG_DIR, transform=train_tfms, is_test=False)
            va_ds = SkinDataset(va_df, TRAIN_IMG_DIR, transform=valid_tfms, is_test=False)
            te_ds = SkinDataset(test_df, TEST_IMG_DIR, transform=valid_tfms, is_test=True)

            tr_loader = build_loader(tr_ds, shuffle=True)
            va_loader = build_loader(va_ds, shuffle=False)
            te_loader = build_loader(te_ds, shuffle=False)

            model = build_model(model_name)
            class_w = make_class_weights(tr_df).to(device)
            criterion = nn.CrossEntropyLoss(weight=class_w, label_smoothing=LABEL_SMOOTHING)

            set_head_only(model, model_name, head_only=True)
            opt_head = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD, weight_decay=WEIGHT_DECAY)

            set_head_only(model, model_name, head_only=False)
            opt_full = AdamW(model.parameters(), lr=LR_FINE, weight_decay=WEIGHT_DECAY)
            sched = CosineAnnealingLR(opt_full, T_max=max(EPOCHS - HEAD_EPOCHS, 1), eta_min=1e-6)

            scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)
            best_f1 = -1.0
            best_state = None
            bad = 0

            for epoch in range(1, EPOCHS + 1):
                model.train()
                losses = []

                if epoch <= HEAD_EPOCHS:
                    set_head_only(model, model_name, head_only=True)
                    optimizer = opt_head
                else:
                    set_head_only(model, model_name, head_only=False)
                    optimizer = opt_full

                for images, labels in tr_loader:
                    images = images.to(device, non_blocking=True)
                    labels = labels.to(device, non_blocking=True)

                    optimizer.zero_grad(set_to_none=True)
                    with torch.amp.autocast('cuda', enabled=USE_AMP):
                        logits = model(images)
                        loss = criterion(logits, labels)

                    scaler.scale(loss).backward()
                    if USE_AMP:
                        scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                    scaler.step(optimizer)
                    scaler.update()
                    losses.append(loss.item())

                if epoch > HEAD_EPOCHS:
                    sched.step()

                val_proba = predict_proba(model, va_loader, use_tta=False)
                val_f1 = macro_f1(y_val, val_proba)
                print(' epoch', epoch, 'loss', round(float(np.mean(losses)), 4), 'val_f1', round(float(val_f1), 4))

                if val_f1 > best_f1:
                    best_f1 = val_f1
                    best_state = copy.deepcopy(model.state_dict())
                    bad = 0
                else:
                    bad += 1

                if bad >= EARLY_STOP:
                    print(' early stop')
                    break

            model.load_state_dict(best_state)

            va_prob_tta = predict_proba(model, va_loader, use_tta=True)
            te_prob_tta = predict_proba(model, te_loader, use_tta=True)
            oof_member[va_idx] = va_prob_tta
            test_member += te_prob_tta / N_SPLITS

            ckpt_path = run_dir / f'member_{model_name}_seed{seed}_fold{fold}.pt'
            torch.save(best_state, ckpt_path)

            member_logs.append({
                'seed': seed,
                'model': model_name,
                'fold': fold,
                'best_val_f1': float(best_f1),
                'ckpt': str(ckpt_path),
            })

            del model, tr_ds, va_ds, te_ds, tr_loader, va_loader, te_loader
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        member_oof_list.append(oof_member)
        member_test_list.append(test_member)
        member_name_list.append(f'{model_name}_seed{seed}')

print('Total members:', len(member_name_list))


In [ ]:
# =========================
# Stage B: blend + stacking + calibration
# =========================
base_weights = []
for _s in SEEDS:
    for m in MODEL_CONFIGS:
        base_weights.append(m['base_weight'])
base_weights = np.array(base_weights, dtype=np.float32)
base_weights = base_weights / base_weights.sum()

opt_w, best_blend_oof = optimize_weights(
    y_true=y_true,
    member_oof_list=member_oof_list,
    base_w=base_weights,
    n_trials=3000,
    seed=777,
)

print('Best OOF macro F1 (weighted blend):', round(best_blend_oof, 6))
for n, w in zip(member_name_list, opt_w):
    print(' ', n, '->', round(float(w), 4))

blend_oof = np.zeros_like(member_oof_list[0])
blend_test = np.zeros_like(member_test_list[0])
for w, poof, ptest in zip(opt_w, member_oof_list, member_test_list):
    blend_oof += w * poof
    blend_test += w * ptest

# Stacking on OOF member probabilities
X_oof = np.concatenate(member_oof_list, axis=1)
X_test = np.concatenate(member_test_list, axis=1)

scaler = StandardScaler()
X_oof_s = scaler.fit_transform(X_oof)
X_test_s = scaler.transform(X_test)

stacker = LogisticRegression(
    max_iter=2500,
    C=1.0,
    class_weight='balanced',
    multi_class='multinomial',
    solver='lbfgs',
)
stacker.fit(X_oof_s, y_true)
stack_oof = stacker.predict_proba(X_oof_s)
stack_test = stacker.predict_proba(X_test_s)

stack_oof_f1 = macro_f1(y_true, stack_oof)
blend_oof_f1 = macro_f1(y_true, blend_oof)
print('OOF blend F1:', round(blend_oof_f1, 6))
print('OOF stack F1:', round(stack_oof_f1, 6))

# Conservative blend of blend+stack
meta_oof = 0.6 * stack_oof + 0.4 * blend_oof
meta_test = 0.6 * stack_test + 0.4 * blend_test
meta_oof_f1 = macro_f1(y_true, meta_oof)
print('OOF meta F1 (pre calibration):', round(meta_oof_f1, 6))

bias_vec, temp_scalar, best_cal = optimize_bias_temp(y_true, meta_oof, rounds=5)
meta_oof_cal = apply_bias_temp(meta_oof, bias_vec, temp_scalar)
meta_test_cal = apply_bias_temp(meta_test, bias_vec, temp_scalar)

print('OOF meta F1 (after bias+temp):', round(best_cal, 6))
print('bias:', bias_vec)
print('temp:', temp_scalar)

pred_oof = np.argmax(meta_oof_cal, axis=1)
print(classification_report(
    y_true,
    pred_oof,
    labels=list(range(N_CLASSES)),
    target_names=CLASS_NAMES,
    digits=4,
    zero_division=0,
))

cm = confusion_matrix(y_true, pred_oof, labels=list(range(N_CLASSES)))
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('OOF Confusion Matrix (Meta Calibrated)')
plt.tight_layout()
plt.savefig(run_dir / 'oof_confusion_matrix_meta_calibrated.png', dpi=180)
plt.show()

final_test_proba = meta_test_cal.copy()


In [ ]:
# =========================
# Stage C: consensus pseudo-label (conservative)
# =========================
if ENABLE_PSEUDO:
    # Member consensus on top-1 class
    member_preds = [np.argmax(p, axis=1) for p in member_test_list]
    member_probs_max = [np.max(p, axis=1) for p in member_test_list]

    consensus = np.zeros(len(test_df), dtype=bool)
    consensus_label = np.full(len(test_df), -1, dtype=int)

    for i in range(len(test_df)):
        votes = [mp[i] for mp in member_preds]
        vals, cnts = np.unique(votes, return_counts=True)
        best_idx = np.argmax(cnts)
        label = vals[best_idx]
        vote_count = cnts[best_idx]

        # require strong agreement + high confidence and margin
        p = final_test_proba[i]
        top1 = p.max()
        p_sorted = np.sort(p)
        margin = p_sorted[-1] - p_sorted[-2]

        if vote_count >= int(np.ceil(0.75 * len(member_preds))) and top1 >= PSEUDO_THRESHOLD and margin >= PSEUDO_MARGIN:
            consensus[i] = True
            consensus_label[i] = int(label)

    n_cons = int(consensus.sum())
    print('Consensus pseudo selected:', n_cons)

    if n_cons >= PSEUDO_MIN_COUNT:
        # prior-aware correction only on selected samples
        prior_train = train_df['label'].value_counts(normalize=True).sort_index().values
        pseudo_counts = np.bincount(consensus_label[consensus], minlength=N_CLASSES).astype(float)
        prior_pseudo = pseudo_counts / pseudo_counts.sum()
        ratio = np.clip(prior_train / np.clip(prior_pseudo, 1e-6, 1.0), 0.6, 1.8)
        ratio = ratio / ratio.mean()

        adjusted = final_test_proba.copy()
        adjusted[consensus] = adjusted[consensus] * ratio[None, :]
        adjusted[consensus] = adjusted[consensus] / adjusted[consensus].sum(axis=1, keepdims=True)

        final_test_proba = (1.0 - PSEUDO_BLEND_ALPHA) * final_test_proba + PSEUDO_BLEND_ALPHA * adjusted

        pseudo_df = test_df.iloc[np.where(consensus)[0]].copy()
        pseudo_df['pseudo_label'] = consensus_label[consensus]
        pseudo_df['pseudo_disease'] = [idx2label[int(x)] for x in pseudo_df['pseudo_label'].values]
        pseudo_df.to_csv(run_dir / 'consensus_pseudo_selected.csv', index=False)
    else:
        print('Consensus pseudo below minimum count; skip pseudo refinement.')


In [ ]:
# =========================
# Stage D: retrieval near-duplicate correction
# =========================
if ENABLE_RETRIEVAL_CORRECTION:
    print('Running retrieval correction...')

    # Use efficientnet_v2_s feature extractor as retrieval encoder
    try:
        retr = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
    except Exception:
        retr = models.efficientnet_v2_s(weights=None)

    # remove classifier, keep global pooled features
    retr.classifier = nn.Identity()
    retr = retr.to(device)
    retr.eval()

    retr_tf = v2.Compose([
        v2.Resize((IMG_SIZE, IMG_SIZE)),
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(MEAN, STD),
    ])

    train_retr_ds = SkinDataset(train_df[['filename', 'label']], TRAIN_IMG_DIR, transform=retr_tf, is_test=False)
    test_retr_ds = SkinDataset(test_df[['id', 'filename']], TEST_IMG_DIR, transform=retr_tf, is_test=True)

    train_retr_loader = build_loader(train_retr_ds, shuffle=False)
    test_retr_loader = build_loader(test_retr_ds, shuffle=False)

    @torch.no_grad()
    def extract_feats(model, loader, is_test=False):
        feats = []
        ids = []
        labels = []
        for batch in loader:
            images = batch[0].to(device, non_blocking=True)
            out = model(images)
            if out.ndim > 2:
                out = F.adaptive_avg_pool2d(out, 1).flatten(1)
            out = F.normalize(out, dim=1)
            feats.append(out.cpu().numpy())
            if is_test:
                ids.extend(batch[1].numpy().tolist())
            else:
                labels.extend(batch[1].numpy().tolist())
        feats = np.concatenate(feats, axis=0)
        return feats, ids, labels

    train_feats, _, train_labels = extract_feats(retr, train_retr_loader, is_test=False)
    test_feats, test_ids, _ = extract_feats(retr, test_retr_loader, is_test=True)
    train_labels = np.array(train_labels)

    # cosine sim via dot product on normalized vectors
    sim = test_feats @ train_feats.T
    nn_idx = np.argmax(sim, axis=1)
    nn_sim = sim[np.arange(len(test_feats)), nn_idx]
    nn_label = train_labels[nn_idx]

    high = nn_sim >= RETRIEVAL_SIM_THRESHOLD
    print('High-sim retrieval matches:', int(high.sum()))

    if int(high.sum()) > 0:
        onehot = np.eye(N_CLASSES, dtype=np.float32)[nn_label]
        corrected = final_test_proba.copy()
        corrected[high] = (1.0 - RETRIEVAL_BLEND) * corrected[high] + RETRIEVAL_BLEND * onehot[high]
        corrected[high] = corrected[high] / corrected[high].sum(axis=1, keepdims=True)
        final_test_proba = corrected

        retr_df = pd.DataFrame({
            'id': test_df['id'].values,
            'nn_train_idx': nn_idx,
            'nn_sim': nn_sim,
            'nn_label': [idx2label[int(x)] for x in nn_label],
            'used': high,
        })
        retr_df.to_csv(run_dir / 'retrieval_matches.csv', index=False)

    del retr, train_retr_ds, test_retr_ds, train_retr_loader, test_retr_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# =========================
# Final submission / logs
# =========================
final_pred = np.argmax(final_test_proba, axis=1)
submission = pd.DataFrame({
    'id': test_df['id'].values,
    'disease': [idx2label[int(i)] for i in final_pred],
}).sort_values('id').reset_index(drop=True)

assert len(submission) == 180
assert submission['disease'].isin(CLASS_NAMES).all()

run_submission_path = run_dir / 'submission.csv'
platform_submission_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(run_submission_path, index=False)
submission.to_csv(platform_submission_path, index=False)

if MODE == 'local':
    local_submission_path = BASE_DIR / 'submission.csv'
    submission.to_csv(local_submission_path, index=False)

pd.DataFrame(member_logs).to_csv(run_dir / 'member_logs.csv', index=False)

summary = {
    'run_name': run_name,
    'mode': MODE,
    'device': str(device),
    'seeds': SEEDS,
    'models': MODEL_CONFIGS,
    'members': member_name_list,
    'opt_member_weights': [float(x) for x in opt_w],
    'oof_blend_before_calibration': float(best_blend_oof),
    'oof_after_calibration': float(best_cal),
    'bias_vector': [float(x) for x in bias_vec],
    'temperature': float(temp_scalar),
    'pseudo_enabled': bool(ENABLE_PSEUDO),
    'pseudo_threshold': float(PSEUDO_THRESHOLD),
    'pseudo_margin': float(PSEUDO_MARGIN),
    'pseudo_min_count': int(PSEUDO_MIN_COUNT),
    'retrieval_enabled': bool(ENABLE_RETRIEVAL_CORRECTION),
    'retrieval_similarity_threshold': float(RETRIEVAL_SIM_THRESHOLD),
    'retrieval_blend': float(RETRIEVAL_BLEND),
}
with open(run_dir / 'run_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('Run dir:', run_dir)
print('Run submission:', run_submission_path)
print('Platform output:', platform_submission_path)
if MODE == 'local':
    print('Local copy:', local_submission_path)

print('Submission distribution:')
print(submission['disease'].value_counts())
submission.head()
